# Train CNN3DMultiAtt

5-fold cross-validation on PI-CAI (T2 + ADC + HBV) using shared SPD Riemannian attention.

In [ ]:
import os
import sys
import pickle
from pathlib import Path

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import torch
import torch.nn as nn
import torch.optim as optim

import config
from data import experiment_name, get_fold_loaders, set_seed
from model import CNN3DMultiAtt

set_seed(config.DEFAULTS["seed"])

GPU = "0"  # set to None to force CPU
if GPU is not None:
    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    print(torch.cuda.get_device_name())

## Hyperparameters

In [ ]:
HP = dict(
    lr=1e-3,
    epochs=15,
    batch_size=32,
    num_classes=2,
    dropout_p=0.0,
    spd_depth=1,
    reduction=1,
    att_position=[1, 0, 1, 0, 1, 0],
    train_ratio=1.0,
    grad_clip=1.0,
    n_folds=5,
    suffix="_v1",
    use_wandb=True,
)

name = experiment_name(
    lr=HP["lr"],
    epochs=HP["epochs"],
    batch_size=HP["batch_size"],
    reduction=HP["reduction"],
    spd_depth=HP["spd_depth"],
    suffix=HP["suffix"],
)
print("Experiment:", name)

run_dir = config.OUTPUT_DIR / CNN3DMultiAtt.model_name / name
weights_path = run_dir / "weights"
results_path = run_dir / "results"
weights_path.mkdir(parents=True, exist_ok=True)
results_path.mkdir(parents=True, exist_ok=True)

## Training loop

In [ ]:
def has_bad_gradients(model):
    for p in model.parameters():
        if p.grad is not None and (torch.isnan(p.grad).any() or torch.isinf(p.grad).any()):
            return True
    return False


wandb = None
if HP["use_wandb"]:
    import wandb as _wandb
    wandb = _wandb
    wandb.login()

for fold in range(HP["n_folds"]):
    run = None
    if wandb is not None:
        run = wandb.init(
            entity=config.DEFAULTS["wandb_entity"],
            project=config.DEFAULTS["wandb_project"],
            name=f"{name}_fold_{fold}",
            config={**HP, "model_name": CNN3DMultiAtt.model_name, "fold": fold},
            reinit=True,
        )

    print(f"\n========== Fold {fold} ==========")
    model = CNN3DMultiAtt(
        num_classes=HP["num_classes"],
        dropout_p=HP["dropout_p"],
        spd_depth=HP["spd_depth"],
        reduction=HP["reduction"],
        att_position=HP["att_position"],
    ).to(device)

    fold_weights = weights_path / f"fold_{fold}.pth"
    fold_results = results_path / f"fold_{fold}.pkl"

    data = get_fold_loaders(
        fold=fold,
        path_data=config.PATH_DATA,
        path_json_info=config.PATH_JSON_INFO,
        path_folds=config.PATH_FOLDS,
        batch_size=HP["batch_size"],
        train_ratio=HP["train_ratio"],
        modality=(1, 1, 1),
    )
    x_train = [data["T2"]["train_loader"], data["ADC"]["train_loader"], data["HBV"]["train_loader"]]
    x_test = [data["T2"]["test_loader"], data["ADC"]["test_loader"], data["HBV"]["test_loader"]]

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=HP["lr"])

    perf_info = []
    nan_total = 0
    min_test_loss = float("inf")

    for epoch in range(HP["epochs"]):
        print(f"\nEpoch {epoch + 1}/{HP['epochs']}  (fold {fold})")

        model.train()
        train_losses = []
        batch_number = 1
        n_train_batches = max(int(len(x_train[0].dataset) / HP["batch_size"]), 1)

        for (lb0, ll0), (lb1, _), (lb2, _) in zip(*x_train):
            logits = model([lb0.to(device), lb1.to(device), lb2.to(device)])
            loss = criterion(logits, ll0.to(device))

            optimizer.zero_grad()
            loss.backward()

            if has_bad_gradients(model):
                nan_total += 1
                if run is not None:
                    run.log({"nan_event": 1, "batch": batch_number})
                print(
                    f"Unstable gradient — fold {fold}, epoch {epoch + 1}, "
                    f"batch {batch_number}. Skipping parameter update."
                )
                batch_number += 1
                continue

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=HP["grad_clip"])
            optimizer.step()

            loss_val = loss.item()
            train_losses.append(loss_val)
            perf_info.append([fold, batch_number, "train", loss_val])
            if batch_number % 5 == 0:
                print(f"  train  batch {batch_number}/{n_train_batches}  loss={loss_val:.4f}")
            batch_number += 1

        model.eval()
        test_losses = []
        batch_number = 1
        n_test_batches = max(int(len(x_test[0].dataset) / HP["batch_size"]), 1)

        with torch.no_grad():
            for (lb0, ll0), (lb1, _), (lb2, _) in zip(*x_test):
                logits = model([lb0.to(device), lb1.to(device), lb2.to(device)])
                loss = criterion(logits, ll0.to(device))
                loss_val = loss.item()
                test_losses.append(loss_val)
                perf_info.append([fold, batch_number, "test", loss_val])
                if batch_number % 5 == 0:
                    print(f"  val    batch {batch_number}/{n_test_batches}  loss={loss_val:.4f}")
                batch_number += 1

        avg_train = sum(train_losses) / len(train_losses) if train_losses else 0.0
        avg_test = sum(test_losses) / len(test_losses) if test_losses else 0.0
        print(f"  avg train={avg_train:.4f}  avg val={avg_test:.4f}  nan_events={nan_total}")

        if run is not None:
            run.log({
                "train_loss": avg_train,
                "test_loss": avg_test,
                "epoch": epoch + 1,
                "total_nans": nan_total,
            })

        if avg_test < min_test_loss:
            min_test_loss = avg_test
            torch.save(model.state_dict(), fold_weights)
            print(f"  checkpoint saved -> {fold_weights}")

    with open(fold_results, "ab") as f:
        pickle.dump(perf_info, f)

    del model, optimizer, x_train, x_test
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if run is not None:
        run.finish()

print(f"\nTraining finished. Checkpoints: {weights_path}")